# Gold → PostgreSQL (AWS Glue / Spark)

Loads gold Parquet tables from S3 into PostgreSQL. Run as single executable cell.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PARAMETERS (edit or pass via Glue job: --PG_URL --PG_USER --PG_PASSWORD)
# ═══════════════════════════════════════════════════════════════════════════════
import sys
import re

try:
    from awsglue.utils import getResolvedOptions
    _a = getResolvedOptions(sys.argv, ["PG_URL", "PG_USER", "PG_PASSWORD"])
    PG_URL, PG_USER, PG_PASSWORD = _a["PG_URL"], _a["PG_USER"], _a["PG_PASSWORD"]
except Exception:
    PG_URL = "jdbc:postgresql://biodiversity-db.cluster-c1mg28uo8tmq.eu-west-2.rds.amazonaws.com:5432/postgres"
    PG_USER = "postgres"
    PG_PASSWORD = "N*A5*P!_K*LUJZmfs2iZ|[1Lq32U"

S3_BUCKET = "ie-datalake"
PG_SCHEMA = "biodiversity_db"
TABLES = ["gbif_cell_metrics", "gbif_species_dim", "gbif_species_h3_mapping", "iucn_species_profiles", "osm_hex_features"]
COUNTRIES = ["ES"]  # None = all
# YEARS = [2024]  # None = all
YEARS = None
H3_RESOLUTIONS = [6, 7, 8, 9]  # None = all
NUM_PARTITIONS = 8
BATCH_SIZE = 10000

# ═══════════════════════════════════════════════════════════════════════════════
# SPARK / GLUE
# ═══════════════════════════════════════════════════════════════════════════════
from pyspark.sql import functions as F

try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext
    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
except ImportError:
    spark = __import__("pyspark.sql", fromlist=["SparkSession"]).SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark.conf.set("spark.sql.files.maxPartitionBytes", "134217728")
spark.conf.set("spark.sql.adaptive.enabled", "true")

# ═══════════════════════════════════════════════════════════════════════════════
# TABLE DEFINITIONS + INDEXES
# ═══════════════════════════════════════════════════════════════════════════════
GOLD_TABLES = {
    "gbif_cell_metrics": {"path": "gold/gbif_cell_metrics", "partitions": ["country", "year", "h3_resolution"], "has_year": True, "has_h3_res": True},
    "gbif_species_dim": {"path": "gold/gbif_species_dim", "partitions": ["country", "year"], "has_year": True, "has_h3_res": False},
    "gbif_species_h3_mapping": {"path": "gold/gbif_species_h3_mapping", "partitions": ["country", "year", "h3_resolution"], "has_year": True, "has_h3_res": True},
    "iucn_species_profiles": {"path": "gold/iucn_species_profiles", "partitions": ["country", "year"], "has_year": True, "has_h3_res": False},
    "osm_hex_features": {"path": "gold/osm_hex_features", "partitions": ["country", "h3_resolution"], "has_year": False, "has_h3_res": True},
}
TABLE_INDEXES = {
    "gbif_cell_metrics": [("idx_gbif_cell_h3", "(h3_index, h3_resolution)", False), ("idx_gbif_cell_res", "(h3_resolution)", False), ("idx_gbif_cell_country_year", "(country, year)", False), ("idx_gbif_cell_species_richness", "(species_richness_cell DESC)", False)],
    "gbif_species_dim": [("idx_species_dim_taxon", "(taxon_key)", False), ("idx_species_dim_country_year", "(country, year)", False)],
    "gbif_species_h3_mapping": [("idx_h3_map_h3", "(h3_index, h3_resolution)", False), ("idx_h3_map_taxon", "(taxon_key)", False), ("idx_h3_map_country_year_res", "(country, year, h3_resolution)", False)],
    "iucn_species_profiles": [("idx_iucn_scientific_name", "(scientific_name)", False), ("idx_iucn_country_year", "(country, year)", False)],
    "osm_hex_features": [("idx_osm_h3", "(h3_index, h3_resolution)", False), ("idx_osm_res", "(h3_resolution)", False), ("idx_osm_country", "(country)", False)],
}


def drop_duplicate_partition_cols(df, table_name):
    cfg = GOLD_TABLES.get(table_name)
    if not cfg:
        return df
    part_lower = {p.lower() for p in cfg["partitions"]}
    seen, to_drop = set(), []
    for c in df.columns:
        cl = c.lower()
        if cl in part_lower:
            if cl in seen:
                to_drop.append(c)
            else:
                seen.add(cl)
    if to_drop:
        df = df.drop(*to_drop)
    return df


def apply_filters(df, table_name):
    cfg = GOLD_TABLES.get(table_name)
    if not cfg:
        return df
    if COUNTRIES and "country" in df.columns:
        df = df.filter(F.col("country").isin(COUNTRIES))
    if YEARS and cfg.get("has_year") and "year" in df.columns:
        df = df.filter(F.col("year").isin(YEARS))
    if H3_RESOLUTIONS and cfg.get("has_h3_res") and "h3_resolution" in df.columns:
        df = df.filter(F.col("h3_resolution").isin(H3_RESOLUTIONS))
    return df


def load_gold_to_df(table_name):
    cfg = GOLD_TABLES.get(table_name)
    if not cfg:
        return None
    base = f"s3://{S3_BUCKET}/{cfg['path']}"
    try:
        df = spark.read.parquet(base)
    except Exception as e:
        print(f"  Skip {table_name}: {e}")
        return None
    df = drop_duplicate_partition_cols(df, table_name)
    return apply_filters(df, table_name)


def write_to_postgres(df, table_name):
    tbl = f"{PG_SCHEMA}.{table_name}"
    df.repartition(NUM_PARTITIONS).write.mode("overwrite").format("jdbc").option("url", PG_URL).option("dbtable", tbl).option("user", PG_USER).option("password", PG_PASSWORD).option("numPartitions", str(NUM_PARTITIONS)).option("batchsize", str(BATCH_SIZE)).save()
    print(f"  Written -> {tbl}")


def ensure_schema():
    """Create schema if not exists."""
    try:
        import psycopg2
        m = re.match(r"jdbc:postgresql://([^:/]+)(?::(\d+))?/(.+)", PG_URL)
        if not m:
            return
        conn = psycopg2.connect(host=m.group(1), port=int(m.group(2) or 5432), dbname=m.group(3).split("?")[0], user=PG_USER, password=PG_PASSWORD)
        cur = conn.cursor()
        cur.execute(f"CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA}")
        conn.commit()
        cur.close()
        conn.close()
        print(f"Schema {PG_SCHEMA} ready")
    except ImportError:
        print(f"(psycopg2 not available; create manually: CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA})")
    except Exception as e:
        print(f"Schema creation: {e}")


def run_create_indexes(table_name):
    indexes = TABLE_INDEXES.get(table_name, [])
    if not indexes:
        return
    try:
        import psycopg2
        m = re.match(r"jdbc:postgresql://([^:/]+)(?::(\d+))?/(.+)", PG_URL)
        if not m:
            print("  Invalid URL; skip indexes")
            return
        conn = psycopg2.connect(host=m.group(1), port=int(m.group(2) or 5432), dbname=m.group(3).split("?")[0], user=PG_USER, password=PG_PASSWORD)
        cur = conn.cursor()
        tbl = f"{PG_SCHEMA}.{table_name}"
        for idx_name, cols, unique in indexes:
            uniq = "UNIQUE " if unique else ""
            cur.execute(f"CREATE INDEX IF NOT EXISTS {idx_name} ON {tbl} {uniq}{cols}")
            print(f"  Created index: {idx_name}")
        conn.commit()
        cur.close()
        conn.close()
    except ImportError:
        print("  (psycopg2 not available; add --additional-python-modules psycopg2-binary)")
    except Exception as e:
        print(f"  Index creation: {e}")


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════
ensure_schema()
for table_name in TABLES:
    if table_name not in GOLD_TABLES:
        print(f"Unknown table: {table_name}")
        continue
    print(f"\n--- {table_name} ---")
    df = load_gold_to_df(table_name)
    if df is None:
        continue
    n = df.count()
    if n == 0:
        print("  No data")
        continue
    print(f"  Rows: {n:,}")
    write_to_postgres(df, table_name)
    run_create_indexes(table_name)
print("\nDone.")